In [ ]:
import os
import sys
from pathlib import Path

# If auto-detect fails, set this manually to your repo root.
REPO_ROOT = None  # e.g. Path('/content/antwerp-port')

def _is_repo_root(path: Path) -> bool:
    return (path / 'training_config.json').exists() and (path / 'env').exists()

def _find_repo_root(start: Path) -> Path | None:
    cur = start
    while True:
        if _is_repo_root(cur):
            return cur
        if cur.parent == cur:
            return None
        cur = cur.parent

def _shallow_search(base: Path, max_depth: int = 3) -> Path | None:
    if not base.exists():
        return None
    base = base.resolve()
    for root, dirs, files in os.walk(base):
        depth = len(Path(root).relative_to(base).parts)
        if depth > max_depth:
            dirs[:] = []
            continue
        root_path = Path(root)
        if _is_repo_root(root_path):
            return root_path
    return None

repo_root = REPO_ROOT
if repo_root is None:
    repo_root = _find_repo_root(Path.cwd())
if repo_root is None:
    # Colab default workdir is often /content
    repo_root = _shallow_search(Path('/content'))

if repo_root is None:
    raise RuntimeError('Could not find repo root. Set REPO_ROOT manually.')

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print('Repo root:', os.getcwd())


In [ ]:
import os
import sys
from pathlib import Path

def _find_repo_root(start: Path) -> Path | None:
    cur = start
    while True:
        if (cur / 'training_config.json').exists() and (cur / 'env').exists():
            return cur
        if cur.parent == cur:
            return None
        cur = cur.parent

repo_root = _find_repo_root(Path.cwd())
if repo_root is not None:
    os.chdir(repo_root)
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))

print('Repo root:', os.getcwd())


# Training Run Notebook

Single training entrypoint for the Antwerp port RL system. This notebook trains a model, saves it, and plots training episode rewards in-memory (no log files).

In [ ]:
import os
import matplotlib.pyplot as plt

from stable_baselines3.common.env_checker import check_env

from env.port_env import PortEnv
from training.config import load_config
from training.env_setup import apply_env_config
from training.eval import evaluate_model
from training.model import train_model

cfg = load_config("training_config.json")
seed = int(cfg["seed"])
total_timesteps = int(cfg["total_timesteps"])
env_cfg = dict(cfg.get("env", {}))
grading_cfg = dict(cfg.get("grading", {}))


In [ ]:
# Check environment contract
env = PortEnv()
apply_env_config(env, env_cfg)
env.reset(seed=seed)
check_env(env)
print("Environment OK")


In [ ]:
# Train and save model
model, _, episode_rewards = train_model(
    cfg, seed_override=seed, timesteps_override=total_timesteps
)

model_save_path = str(cfg["model_save_path"])
os.makedirs(os.path.dirname(model_save_path), exist_ok=True)
model.save(model_save_path)
print("Saved model to:", model_save_path)
print("Episodes recorded:", len(episode_rewards))


In [ ]:
# Plot training episode rewards
if episode_rewards:
    plt.figure(figsize=(8, 4))
    plt.plot(episode_rewards)
    plt.title("Training Episode Reward")
    plt.xlabel("Episode")
    plt.ylabel("Reward")
    plt.tight_layout()
    plt.show()
else:
    print("No episode rewards recorded. Train longer to capture full episodes.")


In [ ]:
# Optional evaluation
if bool(grading_cfg.get("enabled", False)):
    episodes = int(grading_cfg.get("episodes", 10))
    mean_reward, mean_length = evaluate_model(model, seed + 1000, env_cfg, episodes)
    print(f"Mean reward: {mean_reward:.3f}")
    print(f"Mean episode length: {mean_length:.2f}")
